In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('movie-trailer.csv')

In [3]:
print(f'Phim không có trailer: {df["trailer_url"].isnull().sum()}')

Phim không có trailer: 1052


In [4]:
movie=pd.read_csv('movies-new.csv')

In [5]:
movie.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
movie.shape

(9742, 3)

In [7]:
#lấy danh sách các thể loại trong bảng movie không trùng lặp
genre=set()
for i in movie['genres']:
    for j in i.split('|'):
        genre.add(j)

In [8]:
print(genre)

{'Drama', 'Musical', 'Fantasy', 'Documentary', 'Comedy', '(no genres listed)', 'Thriller', 'Sci-Fi', 'Action', 'Children', 'Adventure', 'Animation', 'Horror', 'Western', 'War', 'Mystery', 'Crime', 'Romance', 'IMAX', 'Film-Noir'}


In [9]:
no_genre=movie[movie['genres']=='(no genres listed)']

In [10]:
no_genre.shape

(34, 3)

In [11]:
#xóa các phim không có thể loại và xóa thể loại imax
movie = movie[movie['genres'] != '(no genres listed)']
def remove_imax(genres):
    genre_list = [i for i in genres.split('|') if i != 'IMAX']
    return '|'.join(genre_list)
movie['genres'] = movie['genres'].apply(remove_imax)

In [12]:
# lấy lại danh sách phim sau khi cập nhật
genre=set()
for i in movie['genres']:
    for j in i.split('|'):
        genre.add(j)

In [13]:
genre_list=sorted(genre)
print(genre_list)

['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [14]:
movie.shape

(9708, 3)

In [15]:
def vector(genre, genre_list):
    return [1 if i in genre.split('|') else 0 for i in genre_list]
movie['genre_vector']=movie['genres'].apply(lambda x: vector(x,genre_list))
movie.head()

,movieId,title,genres,genre_vector
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,"[0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
2,3,Grumpier Old Men (1995),Comedy|Romance,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ..."
4,5,Father of the Bride Part II (1995),Comedy,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [16]:
movie['genre_vector'].iloc[1]

[0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [17]:
ratings=pd.read_csv('ratings.csv')

In [18]:
print(ratings['movieId'].nunique())

9724


In [19]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [20]:
merge=movie.merge(ratings,on='movieId')

In [21]:
print(merge['movieId'].nunique())

9690


In [22]:
import numpy as np

In [23]:
def build_vector(user_id,rating,movie,genre_list,threshold=4):
    like_movie=rating[(rating['userId']==user_id) & (rating['rating']>=threshold)]
    if like_movie.empty:
        return pd.Series(np.zeros(len(genre_list)),index=genre_list)
        #lấy các phim đã thích có trong df movie
    like_genres=like_movie.merge(movie[['movieId','genre_vector']],on='movieId')
    if like_genres.empty:
        return pd.Series(np.zeros(len(genre_list)),index=genre_list)
    #tính vector
    vectors=np.stack(like_genres['genre_vector'].to_list())
    cal=vectors.mean(axis=0)
    return pd.Series(cal,index=genre_list)

In [24]:
def recs(profile):
    return profile.sort_values(ascending=False)

In [25]:
# Lọc các phim user 1 thích (ví dụ threshold = 4)
liked = ratings[(ratings['userId'] == 312) & (ratings['rating'] >= 4)]
# Ghép để lấy genre_vector
liked_movies = liked.merge(movie[['movieId','title','genre_vector']], on='movieId')
# Tìm vị trí thể loại
genre = genre_list.index('Comedy')  # phân biệt hoa-thường
# Lọc theo vector (1 là có Animation)
mask = liked_movies['genre_vector'].apply(lambda v: v[genre] == 1)
liked_animation = liked_movies[mask]
# Số lượng phim
liked_animation.shape[0]

28

In [26]:
movie_counts = liked_movies['movieId'].value_counts()
print(f"Số phim đã thích: {len(movie_counts)}")

Số phim đã thích: 140


In [27]:
recs(build_vector(1,ratings,movie,genre_list))

Action         0.380
Adventure      0.370
Comedy         0.350
Drama          0.320
Thriller       0.215
Fantasy        0.205
Crime          0.195
Children       0.185
Sci-Fi         0.160
Animation      0.135
Romance        0.120
War            0.100
Musical        0.100
Mystery        0.065
Horror         0.045
Western        0.030
Film-Noir      0.005
Documentary    0.000
dtype: float64

In [28]:
def calculate(user_profile, movie_vector):
    """Tính điểm tương đồng giữa user profile và phim"""
    return np.dot(user_profile, movie_vector)

def recommend(user_id, user_profiles_df, movies_df, ratings_df, top_n=10):
    # Lấy profile theo nhãn
    user_profile = user_profiles_df.loc[user_id].astype(float)

    # Dựng ma trận thể loại theo đúng genre_list và dùng label alignment
    movie_matrix = pd.DataFrame(movies_df['genre_vector'].tolist(), columns=genre_list)

    # Tính điểm tương đồng theo nhãn cột (không phụ thuộc thứ tự)
    similarity_scores = movie_matrix.mul(user_profile, axis=1).sum(axis=1).to_numpy()

    recommendations = movies_df.copy()
    recommendations['similarity_score'] = similarity_scores

    # Lọc phim đã xem
    seen_movies = set(ratings_df.loc[ratings_df['userId'] == user_id, 'movieId'])
    recommendations = recommendations[~recommendations['movieId'].isin(seen_movies)]

    return recommendations.sort_values('similarity_score', ascending=False).head(top_n)

In [29]:
user=ratings['userId'].unique()
user_profile_df=pd.DataFrame(index=user,columns=genre_list)

In [30]:
for uid in user:
    user_profile_df.loc[uid] = build_vector(uid, ratings, movie, genre_list).values
# Khi user đăng nhập
def get_recommendations_for_user(user_id):
    return recommend(user_id, user_profile_df, movie, ratings, top_n=30)

In [31]:
user_profile_df.head()

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,0.38,0.37,0.135,0.185,0.35,0.195,0.0,0.32,0.205,0.005,0.045,0.1,0.065,0.12,0.16,0.215,0.1,0.03
2,0.368421,0.105263,0.0,0.0,0.263158,0.315789,0.105263,0.578947,0.0,0.0,0.0,0.0,0.105263,0.052632,0.105263,0.315789,0.052632,0.0
3,0.5625,0.25,0.0,0.0,0.0625,0.0,0.0,0.0625,0.125,0.0,0.5,0.0,0.0625,0.0,0.75,0.375,0.0,0.0
4,0.101562,0.148438,0.03125,0.054688,0.453125,0.140625,0.007812,0.546875,0.101562,0.023438,0.03125,0.09375,0.109375,0.226562,0.039062,0.171875,0.039062,0.054688
5,0.130435,0.130435,0.217391,0.304348,0.304348,0.304348,0.0,0.608696,0.217391,0.0,0.0,0.173913,0.043478,0.130435,0.0,0.217391,0.086957,0.043478


In [32]:
get_recommendations_for_user(1)

,movieId,title,genres,genre_vector,similarity_score
7441,81132,Rubber (2010),Action|Adventure|Comedy|Crime|Drama|Film-Noir|...,"[1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, ...",1.975
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,"[1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...",1.840
7170,71999,Aelita: The Queen of Mars (Aelita) (1924),Action|Adventure|Drama|Fantasy|Romance|Sci-Fi|...,"[1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, ...",1.770
3608,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,"[1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",1.755
3460,4719,Osmosis Jones (2001),Action|Animation|Comedy|Crime|Drama|Romance|Th...,"[1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",1.715
4631,6902,Interstate 60 (2002),Adventure|Comedy|Drama|Fantasy|Mystery|Sci-Fi|...,"[0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, ...",1.685
9394,164226,Maximum Ride (2016),Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller,"[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, ...",1.680
6462,52462,Aqua Teen Hunger Force Colon Movie Film for Th...,Action|Adventure|Animation|Comedy|Fantasy|Myst...,"[1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, ...",1.665
478,546,Super Mario Bros. (1993),Action|Adventure|Children|Comedy|Fantasy|Sci-Fi,"[1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, ...",1.650
5476,26236,"White Sun of the Desert, The (Beloe solntse pu...",Action|Adventure|Comedy|Drama|Romance|War,"[1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",1.640


In [33]:
movie.query('movieId == 81132')['genres'].iloc[0]

'Action|Adventure|Comedy|Crime|Drama|Film-Noir|Horror|Mystery|Thriller|Western'

In [34]:
user_profile_df.loc[312]

Action         0.264286
Adventure      0.207143
Animation           0.0
Children       0.007143
Comedy              0.2
Crime              0.15
Documentary    0.007143
Drama          0.485714
Fantasy        0.071429
Film-Noir      0.035714
Horror             0.25
Musical        0.021429
Mystery        0.135714
Romance        0.121429
Sci-Fi             0.35
Thriller       0.335714
War            0.142857
Western        0.014286
Name: 312, dtype: object

In [35]:
merge=movie.merge(ratings,on='movieId')
check="Tommy Boy (1995)" in (merge[merge['userId']==3]['title'].unique())
print(check)

False


In [36]:
print(merge[merge['userId'] == 312]['title'].unique())

['Heat (1995)' 'Casino (1995)' 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)'
 'Braveheart (1995)' 'Taxi Driver (1976)' 'Rob Roy (1995)'
 'Lord of Illusions (1995)' 'Strange Days (1995)'
 'Heavenly Creatures (1994)'
 'Interview with the Vampire: The Vampire Chronicles (1994)'
 'Star Wars: Episode IV - A New Hope (1977)' 'Stargate (1994)'
 'Star Trek: Generations (1994)' 'Backbeat (1993)' 'Forrest Gump (1994)'
 'Wolf (1994)' 'Demolition Man (1993)' 'Fugitive, The (1993)'
 'Hudsucker Proxy, The (1994)' 'Jurassic Park (1993)'
 "Schindler's List (1993)" 'Blade Runner (1982)' 'Son in Law (1993)'
 'Terminator 2: Judgment Day (1991)' 'Fargo (1996)'
 'Mystery Science Theater 3000: The Movie (1996)'
 'Alphaville (Alphaville, une étrange aventure de Lemmy Caution) (1965)'
 'Craft, The (1996)'
 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)'
 'Independence Day (a.k.a. ID4) (1996)' 'Frighteners, The (1996)'
 'Bound (1996)' 'Vertigo (1958)' 'Rear Window (1954)' 'Casablanc

In [37]:
def calculate_movie_similarity(movie_id, movies_df, top_n=10):
    # Kiểm tra xem movie_id có tồn tại không
    if movie_id not in movies_df['movieId'].values:
        print(f"Không tìm thấy phim với movieId: {movie_id}")
        return None
    
    # Lấy genre vector của phim cần tìm
    target_movie = movies_df[movies_df['movieId'] == movie_id].iloc[0]
    target_vector = np.array(target_movie['genre_vector'])
    
    # Tính độ tương đồng với tất cả các phim khác
    similarities = []
    
    for idx, row in movies_df.iterrows():
        if row['movieId'] != movie_id:  # Loại bỏ chính phim đó
            movie_vector = np.array(row['genre_vector'])
            
            # Tính cosine similarity
            dot_product = np.dot(target_vector, movie_vector)
            norm_target = np.linalg.norm(target_vector)
            norm_movie = np.linalg.norm(movie_vector)
            
            if norm_target > 0 and norm_movie > 0:
                similarity = dot_product / (norm_target * norm_movie)
            else:
                similarity = 0
                
            similarities.append({
                'movieId': row['movieId'],
                'title': row['title'],
                'genres': row['genres'],
                'similarity_score': similarity
            })
    similarity_df = pd.DataFrame(similarities)
    similarity_df = similarity_df.sort_values('similarity_score', ascending=False).head(top_n)
    
    return similarity_df

In [38]:
def show_similar_movies(movie_id, movies_df, top_n):
    """
    Hiển thị thông tin phim gốc và danh sách các phim tương tự
    """
    # Lấy thông tin phim gốc
    original_movie = movies_df[movies_df['movieId'] == movie_id]
    if original_movie.empty:
        print(f"Không tìm thấy phim với movieId: {movie_id}")
        return
    
    print("=" * 80)
    print("PHIM GỐC:")
    print(f"ID: {original_movie.iloc[0]['movieId']}")
    print(f"Tên: {original_movie.iloc[0]['title']}")
    print(f"Thể loại: {original_movie.iloc[0]['genres']}")
    print("=" * 80)
    
    # Tìm các phim tương tự
    similar_movies = calculate_movie_similarity(movie_id, movies_df, top_n)
    
    if similar_movies is not None:
        print(f"\nTOP {top_n} PHIM TƯƠNG TỰ:")
        print("-" * 80)
        
        for idx, row in similar_movies.iterrows():
            print(f"{row.name + 1:2d}. {row['title']}")
            print(f"    ID: {row['movieId']} | Thể loại: {row['genres']}")
            print(f"    Độ tương đồng: {row['similarity_score']:.4f}")
            print()

In [40]:
show_similar_movies(4956,movie,10)

PHIM GỐC:
ID: 4956
Tên: Stunt Man, The (1980)
Thể loại: Action|Adventure|Comedy|Drama|Romance|Thriller

TOP 10 PHIM TƯƠNG TỰ:
--------------------------------------------------------------------------------
4445. Lara Croft Tomb Raider: The Cradle of Life (2003)
    ID: 6564 | Thể loại: Action|Adventure|Comedy|Romance|Thriller
    Độ tương đồng: 0.9129

6094. Casanova (2005)
    ID: 42015 | Thể loại: Action|Adventure|Comedy|Drama|Romance
    Độ tương đồng: 0.9129

338. True Lies (1994)
    ID: 380 | Thể loại: Action|Adventure|Comedy|Romance|Thriller
    Độ tương đồng: 0.9129

3513. King Solomon's Mines (1937)
    ID: 4800 | Thể loại: Action|Adventure|Drama|Romance|Thriller
    Độ tương đồng: 0.9129

6570. Hunting Party, The (2007)
    ID: 55116 | Thể loại: Action|Adventure|Comedy|Drama|Thriller
    Độ tương đồng: 0.9129

401. Getaway, The (1994)
    ID: 459 | Thể loại: Action|Adventure|Crime|Drama|Romance|Thriller
    Độ tương đồng: 0.8333

5476. White Sun of the Desert, The (Beloe sol